In [28]:
api_key = 'WxRSAcpuv90pVAdgPMmmQAAely5rpsJ6'

In [36]:
pip install ta

  Preparing metadata (setup.py) ... done
  Created wheel for ta: filename=ta-0.11.0-py3-none-any.whl size=29412 sha256=31422a38bb01e141b081d30443e8d4650a8c132f2f26670c2b040bd21e0fb47f
  Stored in directory: /root/.cache/pip/wheels/a1/d7/29/7781cc5eb9a3659d032d7d15bdd0f49d07d2b24fec29f44bc4
Successfully built ta


In [64]:
import pandas as pd
import numpy as np
import pandas_ta as ta
import requests
import os
from datetime import datetime

# FMP API setup
FMP_API_KEY = os.getenv('FMP_API_KEY', 'WxRSAcpuv90pVAdgPMmmQAAely5rpsJ6')  # Replace with your FMP API key
FMP_BASE_URL = 'https://financialmodelingprep.com/api/v3'

# Fetch SPY data from FMP
def fetch_fmp_data(symbol, start_date, end_date):
    url = f"{FMP_BASE_URL}/historical-price-full/{symbol}?from={start_date}&to={end_date}&apikey={FMP_API_KEY}"
    response = requests.get(url)
    data = response.json()
    if 'historical' not in data:
        raise ValueError("Error fetching data from FMP. Check API key or symbol.")
    df = pd.DataFrame(data['historical'])
    df['date'] = pd.to_datetime(df['date'])
    df = df[['date', 'open', 'high', 'low', 'close', 'volume']]
    df.columns = ['Date', 'Open', 'High', 'Low', 'Close', 'Volume']
    df.set_index('Date', inplace=True)
    df.sort_index(inplace=True)
    return df

# Load data
data = fetch_fmp_data('XAUUSD', '2010-01-01', '2024-12-31')
data['Returns'] = data['Close'].pct_change()

# Calculate diversified indicators using pandas_ta
# Trend Indicators
data['SMA_50'] = ta.sma(data['Close'], length=50)
data['SMA_200'] = ta.sma(data['Close'], length=200)
data['EMA_12'] = ta.ema(data['Close'], length=12)
data['EMA_26'] = ta.ema(data['Close'], length=26)
data['PSAR'] = ta.psar(data['High'], data['Low'], data['Close'])['PSARl_0.02_0.2']
ichimoku = ta.ichimoku(data['High'], data['Low'], data['Close'])
data['Ichimoku_A'] = ichimoku[0]['ISA_9']
data['Ichimoku_B'] = ichimoku[0]['ISB_26']
adx = ta.adx(data['High'], data['Low'], data['Close'], length=14)
data['ADX'] = adx['ADX_14']
data['ADX_pos'] = adx['DMP_14']
data['ADX_neg'] = adx['DMN_14']

# Momentum Indicators
data['ROC'] = ta.roc(data['Close'], length=12)
data['MOM'] = ta.mom(data['Close'], length=10)
data['CCI'] = ta.cci(data['High'], data['Low'], data['Close'], length=20)
tsi = ta.tsi(data['Close'])
data['TSI'] = tsi['TSI_13_25_13']
data['TSI_Signal'] = tsi['TSIs_13_25_13']
kvo = ta.kvo(data['High'], data['Low'], data['Close'], data['Volume'])
data['KVO'] = kvo['KVO_34_55_13']
data['KVO_Signal'] = kvo['KVOs_34_55_13']

# Volume Indicators
data['OBV'] = ta.obv(data['Close'], data['Volume'])
data['CMF'] = ta.cmf(data['High'], data['Low'], data['Close'], data['Volume'], length=20)
data['AD'] = ta.ad(data['High'], data['Low'], data['Close'], data['Volume'])
# data['VWAP'] = ta.vwap(data['High'], data['Low'], data['Close'], data['Volume'])['VWAP_D']
# data['FI'] = ta.fi(data['Close'], data['Volume'], length=13)

# Oscillator Indicators
data['RSI'] = ta.rsi(data['Close'], length=14)
stoch = ta.stoch(data['High'], data['Low'], data['Close'])
data['Stoch_K'] = stoch['STOCHk_14_3_3']
data['Williams_R'] = ta.willr(data['High'], data['Low'], data['Close'], length=14)
data['MFI'] = ta.mfi(data['High'], data['Low'], data['Close'], data['Volume'], length=14)
data['UO'] = ta.uo(data['High'], data['Low'], data['Close'])

# Signal generation
signals = pd.DataFrame(index=data.index)
signals['SMA_Crossover'] = np.where(data['SMA_50'] > data['SMA_200'], 1, -1)
signals['EMA_Crossover'] = np.where(data['EMA_12'] > data['EMA_26'], 1, -1)
signals['PSAR'] = np.where(data['Close'] > data['PSAR'], 1, -1)
signals['Ichimoku'] = np.where(data['Close'] > (data['Ichimoku_A'] + data['Ichimoku_B']) / 2, 1,
                              np.where(data['Close'] < (data['Ichimoku_A'] + data['Ichimoku_B']) / 2, -1, 0))
signals['ADX'] = np.where((data['ADX_pos'] > data['ADX_neg']) & (data['ADX'] > 20), 1,
                         np.where((data['ADX_neg'] > data['ADX_pos']) & (data['ADX'] > 20), -1, 0))
signals['ROC'] = np.where(data['ROC'] > 0, 1, -1)
signals['MOM'] = np.where(data['MOM'] > 0, 1, -1)
signals['CCI'] = np.where(data['CCI'] < -100, 1, np.where(data['CCI'] > 100, -1, 0))
signals['TSI'] = np.where(data['TSI'] > data['TSI_Signal'], 1, -1)
signals['KVO'] = np.where(data['KVO'] > data['KVO_Signal'], 1, -1)
signals['OBV'] = np.where(data['OBV'].diff() > 0, 1, -1)
signals['CMF'] = np.where(data['CMF'] > 0, 1, -1)
signals['AD'] = np.where(data['AD'].diff() > 0, 1, -1)
# signals['VWAP'] = np.where(data['Close'] > data['VWAP'], 1, -1)
# signals['FI'] = np.where(data['FI'] > 0, 1, -1)
signals['RSI'] = np.where(data['RSI'] < 30, 1, np.where(data['RSI'] > 70, -1, 0))
signals['Stoch'] = np.where(data['Stoch_K'] < 20, 1, np.where(data['Stoch_K'] > 80, -1, 0))
signals['Williams_R'] = np.where(data['Williams_R'] < -80, 1, np.where(data['Williams_R'] > -20, -1, 0))
signals['MFI'] = np.where(data['MFI'] < 20, 1, np.where(data['MFI'] > 80, -1, 0))
signals['UO'] = np.where(data['UO'] < 30, 1, np.where(data['UO'] > 70, -1, 0))

# Aggregate signals
signals['Total'] = signals.sum(axis=1)
signals['Position'] = np.where(signals['Total'] >= 5, 1, np.where(signals['Total'] <= -5, -1, 0))

# Backtest
data['Strategy_Returns'] = data['Returns'] * signals['Position'].shift(1)
data['Trade'] = signals['Position'].diff().abs()
data['Cost'] = data['Trade'] * 0.001  # 0.1% transaction cost
data['Strategy_Returns'] -= data['Cost']
data['Portfolio'] = (1 + data['Strategy_Returns']).cumprod()
data['Buy_Hold'] = (1 + data['Returns']).cumprod()

# Performance metrics
def calculate_metrics(returns, portfolio):
    annualized_return = ((portfolio.iloc[-1]) ** (252 / len(portfolio)) - 1) * 100
    sharpe_ratio = returns.mean() * 252 / (returns.std() * np.sqrt(252)) if returns.std() != 0 else 0
    max_drawdown = ((portfolio.cummax() - portfolio) / portfolio.cummax()).max() * 100
    win_rate = (returns > 0).mean() * 100 if len(returns) > 0 else 0
    return annualized_return, sharpe_ratio, max_drawdown, win_rate

strat_ret, strat_sharpe, strat_mdd, strat_win = calculate_metrics(data['Strategy_Returns'].dropna(), data['Portfolio'])
bh_ret, bh_sharpe, bh_mdd, bh_win = calculate_metrics(data['Returns'].dropna(), data['Buy_Hold'])

# Output results
print("Strategy Performance:")
print(f"Annualized Return: {strat_ret:.2f}%")
print(f"Sharpe Ratio: {strat_sharpe:.2f}")
print(f"Max Drawdown: {strat_mdd:.2f}%")
print(f"Win Rate: {strat_win:.2f}%")
print("\nBuy-and-Hold Performance:")
print(f"Annualized Return: {bh_ret:.2f}%")
print(f"Sharpe Ratio: {bh_sharpe:.2f}")
print(f"Max Drawdown: {bh_mdd:.2f}%")
print(f"Win Rate: {bh_win:.2f}%")

Strategy Performance:
Annualized Return: -8.77%
Sharpe Ratio: -0.78
Max Drawdown: 76.24%
Win Rate: 22.10%

Buy-and-Hold Performance:
Annualized Return: 5.63%
Sharpe Ratio: 0.44
Max Drawdown: 44.69%
Win Rate: 52.32%


<ipython-input-64-31733614a3b0>:69: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[2.81480795e+07 3.00884667e+07 2.80416462e+07 ... 1.00684627e+09
 1.55842731e+06 8.50428872e+08]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  data['MFI'] = ta.mfi(data['High'], data['Low'], data['Close'], data['Volume'], length=14)
<ipython-input-64-31733614a3b0>:69: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[3.49859895e+07 2.65788000e+07 3.04574265e+07 ... 5.29072868e+08
 8.08753494e+08 1.06391111e+09]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  data['MFI'] = ta.mfi(data['High'], data['Low'], data['Close'], data['Volume'], length=14)
